# 04. Bahdanau Attention (Bahdanau et al., 2014)

Interactive walkthrough accompanying `04_bahdanau_attention.py`.

We cover:
1. Why fixed-length encoder-decoder models hit an information bottleneck on long sequences ($T \ge 20$).
2. **Bidirectional GRU Encoder** annotations $h_j = [\vec{h}_j; \overleftarrow{h}_j]$ from raw tensor ops.
3. **Bahdanau Additive Attention** ($e_{ij}, \alpha_{ij}, c_i$) and an **Attentional Decoder** conditioning on $[y_{i-1}; c_i]$.
4. Shape, simplex conservation, padding-mask, and direct encoder-gradient checks.
5. Training curves, length-stratified greedy accuracy ($T = 10 \dots 26$) vs. a fixed-bottleneck baseline, and an alignment heatmap $\alpha_{ij}$.

## 1. Setup

Import PyTorch and Matplotlib, and define special vocabulary indices.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

PAD_IDX, SOS_IDX, EOS_IDX = 0, 1, 2
CONTENT_START = 3

torch.manual_seed(42)
print("PyTorch version:", torch.__version__)

## 2. Raw-Tensor GRU & Bidirectional Encoder

### Equations
At step $t$, the GRU cell (Cho et al., 2014; Bahdanau et al., 2014 Appendix A.2) updates hidden state $h_t$ using update gate $z_t$ and reset gate $r_t$:

$$z_t = \sigma(x_t W_{xz} + h_{t-1} U_{hz} + b_z), \qquad r_t = \sigma(x_t W_{xr} + h_{t-1} U_{hr} + b_r)$$

$$\tilde{h}_t = \tanh(x_t W_{xh} + (r_t \odot h_{t-1}) U_{hh} + b_h)$$

$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

Running one GRU forward ($j = 0 \dots T_x - 1$) and one backward ($j = T_x - 1 \dots 0$) yields bidirectional annotations and initial decoder state:

$$h_j = \begin{bmatrix} \vec{h}_j \\ \overleftarrow{h}_j \end{bmatrix} \in \mathbb{R}^{2 H_{\text{enc}}}, \qquad s_0 = \tanh(W_s \overleftarrow{h}_0 + b_s)$$

### Intuition
A unidirectional encoder only encodes prefix context $x_{\le j}$ at position $j$. Concatenating $\vec{h}_j$ and $\overleftarrow{h}_j$ ensures every annotation $h_j$ summarizes both the preceding and following tokens around source position $j$. Masking padded timesteps holds the backward state constant over trailing `<PAD>` tokens so $\overleftarrow{h}_0$ stays uncorrupted.

In [ ]:
class GRUCellFromScratch(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.W_xzr = nn.Parameter(torch.empty(input_dim, 2 * hidden_dim))
        self.U_hzr = nn.Parameter(torch.empty(hidden_dim, 2 * hidden_dim))
        self.b_zr = nn.Parameter(torch.zeros(2 * hidden_dim))
        self.W_xh = nn.Parameter(torch.empty(input_dim, hidden_dim))
        self.U_hh = nn.Parameter(torch.empty(hidden_dim, hidden_dim))
        self.b_h = nn.Parameter(torch.zeros(hidden_dim))
        bound = 1.0 / (hidden_dim ** 0.5)
        for p in (self.W_xzr, self.U_hzr, self.W_xh, self.U_hh):
            nn.init.uniform_(p, -bound, bound)

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor | None = None) -> torch.Tensor:
        if h_prev is None:
            h_prev = torch.zeros(x_t.shape[0], self.hidden_dim, device=x_t.device, dtype=x_t.dtype)
        zr = torch.sigmoid(x_t @ self.W_xzr + h_prev @ self.U_hzr + self.b_zr)
        z_t, r_t = zr.chunk(2, dim=-1)
        h_tilde = torch.tanh(x_t @ self.W_xh + (r_t * h_prev) @ self.U_hh + self.b_h)
        return (1.0 - z_t) * h_prev + z_t * h_tilde


class BidirectionalGRUEncoder(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, enc_dim: int, dec_dim: int):
        super().__init__()
        self.enc_dim = enc_dim
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.fwd_cell = GRUCellFromScratch(emb_dim, enc_dim)
        self.bwd_cell = GRUCellFromScratch(emb_dim, enc_dim)
        self.init_decoder = nn.Linear(enc_dim, dec_dim)

    def forward(self, src: torch.Tensor, src_mask: torch.Tensor | None = None):
        B, T_x = src.shape
        x = self.embedding(src)
        if src_mask is None:
            src_mask = torch.ones(B, T_x, dtype=torch.bool, device=src.device)

        h_fwd = torch.zeros(B, self.enc_dim, device=src.device, dtype=x.dtype)
        fwd_states = []
        for j in range(T_x):
            m_j = src_mask[:, j : j + 1].to(x.dtype)
            h_fwd = m_j * self.fwd_cell(x[:, j], h_fwd) + (1.0 - m_j) * h_fwd
            fwd_states.append(h_fwd)

        h_bwd = torch.zeros(B, self.enc_dim, device=src.device, dtype=x.dtype)
        bwd_states = [h_bwd] * T_x
        for j in range(T_x - 1, -1, -1):
            m_j = src_mask[:, j : j + 1].to(x.dtype)
            h_bwd = m_j * self.bwd_cell(x[:, j], h_bwd) + (1.0 - m_j) * h_bwd
            bwd_states[j] = h_bwd

        annotations = torch.cat(
            [torch.stack(fwd_states, dim=1), torch.stack(bwd_states, dim=1)], dim=-1
        )
        s0 = torch.tanh(self.init_decoder(bwd_states[0]))
        return annotations, s0

print("GRUCellFromScratch and BidirectionalGRUEncoder ready.")

## 3. Bahdanau Additive Attention & Attentional Decoder

### Equations
At each decoder output step $i$, given previous decoder state $s_{i-1}$ and encoder annotations $\{h_1, \dots, h_{T_x}\}$:

1. **Alignment energy** (additive compatibility score):
   $$e_{ij} = v_a^\top \tanh(W_a s_{i-1} + U_a h_j)$$
2. **Attention weights** (simplex distribution over valid source tokens; `<PAD>` masked to $-10^9$):
   $$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k=1}^{T_x} \exp(e_{ik})}$$
3. **Context vector** (expected annotation):
   $$c_i = \sum_{j=1}^{T_x} \alpha_{ij} h_j$$
4. **Decoder recurrence & readout** conditioned on $[y_{i-1}; c_i]$:
   $$s_i = \text{GRU}([y_{i-1}; c_i], s_{i-1}), \qquad \text{logits}_i = W_o [s_i; y_{i-1}; c_i] + b_o$$

### Intuition
Instead of forcing the encoder to pack an entire $T_x$-token sentence into one fixed vector, the decoder computes a dynamic query $W_a s_{i-1}$ at every step $i$ and reads out the relevant source annotations via $\alpha_{i\cdot}$. Because $U_a h_j$ depends only on the encoder annotations, it can be precomputed once per sequence.

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, dec_dim: int, ann_dim: int, attn_dim: int):
        super().__init__()
        self.W_a = nn.Linear(dec_dim, attn_dim, bias=False)
        self.U_a = nn.Linear(ann_dim, attn_dim, bias=False)
        self.v_a = nn.Linear(attn_dim, 1, bias=False)

    def precompute_keys(self, annotations: torch.Tensor) -> torch.Tensor:
        return self.U_a(annotations)

    def forward(self, s_prev, annotations, src_mask=None, keys=None):
        if keys is None:
            keys = self.precompute_keys(annotations)
        query = self.W_a(s_prev).unsqueeze(1)
        energies = self.v_a(torch.tanh(query + keys)).squeeze(-1)
        if src_mask is not None:
            energies = energies.masked_fill(~src_mask, -1e9)
        alpha = F.softmax(energies, dim=-1)
        context = torch.bmm(alpha.unsqueeze(1), annotations).squeeze(1)
        return alpha, context


class BahdanauDecoder(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, enc_dim: int, dec_dim: int, attn_dim: int):
        super().__init__()
        ann_dim = 2 * enc_dim
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.attention = BahdanauAttention(dec_dim, ann_dim, attn_dim)
        self.cell = GRUCellFromScratch(emb_dim + ann_dim, dec_dim)
        self.readout = nn.Linear(dec_dim + emb_dim + ann_dim, vocab_size)

    def step(self, y_prev, s_prev, annotations, src_mask=None, keys=None):
        y_emb = self.embedding(y_prev)
        alpha, context = self.attention(s_prev, annotations, src_mask, keys)
        s_i = self.cell(torch.cat([y_emb, context], dim=-1), s_prev)
        logits = self.readout(torch.cat([s_i, y_emb, context], dim=-1))
        return logits, s_i, alpha


class BahdanauSeq2Seq(nn.Module):
    def __init__(self, vocab_size: int, emb_dim=32, enc_dim=48, dec_dim=48, attn_dim=48):
        super().__init__()
        self.encoder = BidirectionalGRUEncoder(vocab_size, emb_dim, enc_dim, dec_dim)
        self.decoder = BahdanauDecoder(vocab_size, emb_dim, enc_dim, dec_dim, attn_dim)

    def forward(self, src, tgt_in, src_mask=None, retain_annotations=False):
        annotations, s = self.encoder(src, src_mask)
        if retain_annotations and annotations.requires_grad:
            annotations.retain_grad()
        keys = self.decoder.attention.precompute_keys(annotations)
        logits_list, alpha_list = [], []
        for i in range(tgt_in.shape[1]):
            logits, s, alpha = self.decoder.step(tgt_in[:, i], s, annotations, src_mask, keys)
            logits_list.append(logits)
            alpha_list.append(alpha)
        return torch.stack(logits_list, dim=1), torch.stack(alpha_list, dim=1), annotations

    @torch.no_grad()
    def decode_greedy(self, src, src_mask, max_len: int):
        annotations, s = self.encoder(src, src_mask)
        keys = self.decoder.attention.precompute_keys(annotations)
        y_prev = torch.full((src.shape[0],), SOS_IDX, dtype=torch.long, device=src.device)
        preds, alphas = [], []
        for _ in range(max_len):
            logits, s, alpha = self.decoder.step(y_prev, s, annotations, src_mask, keys)
            y_prev = logits.argmax(dim=-1)
            preds.append(y_prev)
            alphas.append(alpha)
        return torch.stack(preds, dim=1), torch.stack(alphas, dim=1)

print("BahdanauAttention and BahdanauSeq2Seq ready.")

## 4. Shape, Simplex Conservation & Direct Gradient Flow Checks

Verify:
1. All output and attention shapes match `[B, T_y, V]` and `[B, T_y, T_x]`.
2. Attention weights satisfy $\sum_j \alpha_{ij} = 1$ and place zero probability on `<PAD>` positions.
3. A scalar loss at decoder step $i = 0$ sends non-zero gradients directly back to encoder annotations $h_j$ across valid source positions.

In [ ]:
B, T_x, T_y, V = 4, 12, 10, 15
check_model = BahdanauSeq2Seq(vocab_size=V, emb_dim=16, enc_dim=24, dec_dim=24, attn_dim=20)

src_test = torch.randint(CONTENT_START, V, (B, T_x))
src_test[:, -3:] = PAD_IDX
mask_test = src_test != PAD_IDX
tgt_test = torch.randint(CONTENT_START, V, (B, T_y))
tgt_test[:, 0] = SOS_IDX

logits_t, attn_t, anns_t = check_model(src_test, tgt_test, mask_test, retain_annotations=True)
assert logits_t.shape == (B, T_y, V)
assert attn_t.shape == (B, T_y, T_x)
assert torch.allclose(attn_t.sum(dim=-1), torch.ones(B, T_y), atol=1e-6)
assert attn_t[:, :, -3:].abs().max().item() < 1e-6

logits_t[:, 0, :].sum().backward()
grad_per_pos = anns_t.grad.norm(dim=-1).mean(dim=0)
assert grad_per_pos[:-3].mean().item() > 0.0

print(f"logits={tuple(logits_t.shape)} | attn={tuple(attn_t.shape)} | "
      f"max_sum_err={(attn_t.sum(dim=-1) - 1.0).abs().max().item():.2e} | "
      f"pad_attn_max={attn_t[:, :, -3:].abs().max().item():.2e}")
print("Mean ||dL_0 / dh_j|| (valid j=0..8):", [round(v, 4) for v in grad_per_pos[:-3].tolist()])

## 5. Fixed-Bottleneck Seq2Seq vs. Bahdanau Attention on Long Sequences ($T = 20+$)

Build a fixed-vector GRU encoder-decoder (`FixedBottleneckSeq2Seq`, `hidden_dim=64`) using the same `GRUCellFromScratch` primitive, train both models on variable-length sequence reversal ($T \in [6, 24]$ with `<SOS>`, `<EOS>`, `<PAD>`), and evaluate **greedy decoding** across $T \in \{10, 16, 20, 24, 26\}$.

In [ ]:
class FixedBottleneckSeq2Seq(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int = 32, hidden_dim: int = 64):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.src_embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.enc_cell = GRUCellFromScratch(emb_dim, hidden_dim)
        self.dec_embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.dec_cell = GRUCellFromScratch(emb_dim, hidden_dim)
        self.readout = nn.Linear(hidden_dim + emb_dim, vocab_size)

    def encode(self, src, src_mask):
        B, T_x = src.shape
        x = self.src_embedding(src)
        h = torch.zeros(B, self.hidden_dim, device=src.device, dtype=x.dtype)
        for j in range(T_x):
            m_j = src_mask[:, j : j + 1].to(x.dtype)
            h = m_j * self.enc_cell(x[:, j], h) + (1.0 - m_j) * h
        return h

    def forward(self, src, tgt_in, src_mask):
        s = self.encode(src, src_mask)
        logits_list = []
        for i in range(tgt_in.shape[1]):
            y_emb = self.dec_embedding(tgt_in[:, i])
            s = self.dec_cell(y_emb, s)
            logits_list.append(self.readout(torch.cat([s, y_emb], dim=-1)))
        return torch.stack(logits_list, dim=1)

    @torch.no_grad()
    def decode_greedy(self, src, src_mask, max_len: int):
        s = self.encode(src, src_mask)
        y_prev = torch.full((src.shape[0],), SOS_IDX, dtype=torch.long, device=src.device)
        preds = []
        for _ in range(max_len):
            y_emb = self.dec_embedding(y_prev)
            s = self.dec_cell(y_emb, s)
            y_prev = self.readout(torch.cat([s, y_emb], dim=-1)).argmax(dim=-1)
            preds.append(y_prev)
        return torch.stack(preds, dim=1)


def sample_reversal_batch(batch_size: int, min_len: int, max_len: int, vocab_size: int):
    lengths = torch.randint(min_len, max_len + 1, (batch_size,))
    max_t = int(lengths.max().item()) + 1
    src = torch.full((batch_size, max_t), PAD_IDX, dtype=torch.long)
    tgt_in = torch.full((batch_size, max_t), PAD_IDX, dtype=torch.long)
    tgt_out = torch.full((batch_size, max_t), PAD_IDX, dtype=torch.long)
    for b in range(batch_size):
        L = int(lengths[b].item())
        seq = torch.randint(CONTENT_START, vocab_size, (L,))
        rev = seq.flip(0)
        src[b, :L] = seq
        src[b, L] = EOS_IDX
        tgt_in[b, 0] = SOS_IDX
        tgt_in[b, 1 : L + 1] = rev
        tgt_out[b, :L] = rev
        tgt_out[b, L] = EOS_IDX
    return src, tgt_in, tgt_out, src != PAD_IDX


def evaluate_greedy(model, vocab_size: int, seq_len: int, n_samples: int = 256):
    model.eval()
    src, _, tgt_out, mask = sample_reversal_batch(n_samples, seq_len, seq_len, vocab_size)
    out = model.decode_greedy(src, mask, max_len=tgt_out.shape[1])
    preds = out[0] if isinstance(out, tuple) else out
    valid = tgt_out != PAD_IDX
    tok_acc = ((preds == tgt_out) & valid).sum().item() / valid.sum().item()
    seq_acc = (((preds == tgt_out) | ~valid).all(dim=1)).float().mean().item()
    model.train()
    return tok_acc, seq_acc

torch.manual_seed(42)
vocab_size = 15
steps = 360
bahdanau = BahdanauSeq2Seq(vocab_size=vocab_size, emb_dim=32, enc_dim=48, dec_dim=48, attn_dim=48)
fixed = FixedBottleneckSeq2Seq(vocab_size=vocab_size, emb_dim=32, hidden_dim=64)

opt_b = torch.optim.Adam(bahdanau.parameters(), lr=5e-3)
opt_f = torch.optim.Adam(fixed.parameters(), lr=5e-3)
hist_b, hist_f = [], []

for step in range(1, steps + 1):
    src, tin, tout, mask = sample_reversal_batch(64, 6, 24, vocab_size)

    opt_b.zero_grad()
    lb, _, _ = bahdanau(src, tin, mask)
    loss_b = F.cross_entropy(lb.view(-1, vocab_size), tout.view(-1), ignore_index=PAD_IDX)
    loss_b.backward()
    nn.utils.clip_grad_norm_(bahdanau.parameters(), 1.0)
    opt_b.step()
    hist_b.append(loss_b.item())

    opt_f.zero_grad()
    lf = fixed(src, tin, mask)
    loss_f = F.cross_entropy(lf.view(-1, vocab_size), tout.view(-1), ignore_index=PAD_IDX)
    loss_f.backward()
    nn.utils.clip_grad_norm_(fixed.parameters(), 1.0)
    opt_f.step()
    hist_f.append(loss_f.item())

eval_lengths = [10, 16, 20, 24, 26]
tok_fixed, seq_fixed, tok_bah, seq_bah = [], [], [], []
for T in eval_lengths:
    tf, sf = evaluate_greedy(fixed, vocab_size, T)
    tb, sb = evaluate_greedy(bahdanau, vocab_size, T)
    tok_fixed.append(tf); seq_fixed.append(sf)
    tok_bah.append(tb); seq_bah.append(sb)
    print(f"T={T:2d} | Fixed tok={tf:.3f}, seq={sf:.3f} | Bahdanau tok={tb:.3f}, seq={sb:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_f, label="Fixed-Bottleneck GRU (H=64)", color="#e74c3c", alpha=0.85)
axes[0].plot(hist_b, label="Bahdanau Attention (BiGRU + Attn)", color="#2980b9", alpha=0.9)
axes[0].set_title("Training Cross-Entropy Loss (T in [6, 24])")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.4)
axes[0].legend()

axes[1].plot(eval_lengths, tok_bah, "o-", label="Bahdanau Token Acc", color="#2980b9", lw=2)
axes[1].plot(eval_lengths, seq_bah, "s--", label="Bahdanau Exact Seq Acc", color="#1b4f72", lw=2)
axes[1].plot(eval_lengths, tok_fixed, "o-", label="Fixed-Bottleneck Token Acc", color="#e74c3c", lw=2)
axes[1].plot(eval_lengths, seq_fixed, "s--", label="Fixed-Bottleneck Exact Seq Acc", color="#922b21", lw=2)
axes[1].axvline(24, color="gray", linestyle=":", label="Max Train Length (24)")
axes[1].set_title("Greedy Decoding Accuracy vs. Sequence Length")
axes[1].set_xlabel("Sequence Length T")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(True, alpha=0.4)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Alignment Attention Heatmap ($\alpha_{ij}$)

Run greedy decoding on a single long sequence ($T = 20$ content tokens + `<EOS>`) and plot the learned alignment matrix $\alpha_{ij}$ over source tokens (columns) vs. predicted target tokens (rows).

In [ ]:
bahdanau.eval()
src_one, _, tgt_one, mask_one = sample_reversal_batch(1, 20, 20, vocab_size)
with torch.no_grad():
    preds_one, attn_one = bahdanau.decode_greedy(src_one, mask_one, max_len=tgt_one.shape[1])

def tok_label(idx: int) -> str:
    if idx == PAD_IDX:
        return "<PAD>"
    if idx == SOS_IDX:
        return "<SOS>"
    if idx == EOS_IDX:
        return "<EOS>"
    return f"t{idx - CONTENT_START}"

src_labels = [tok_label(int(i)) for i in src_one[0].tolist()]
pred_labels = [tok_label(int(i)) for i in preds_one[0].tolist()]
exact_match = torch.equal(preds_one[0], tgt_one[0])
print(f"T=20 Greedy exact match: {exact_match}")
print("Source:   ", " ".join(src_labels))
print("Predicted:", " ".join(pred_labels))

alpha_mat = attn_one[0].numpy()
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(alpha_mat, cmap="viridis", aspect="auto", vmin=0.0, vmax=1.0)
ax.set_xticks(range(len(src_labels)))
ax.set_xticklabels(src_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(pred_labels)))
ax.set_yticklabels(pred_labels, fontsize=8)
ax.set_xlabel("Source Position j")
ax.set_ylabel("Decoder Output Step i")
ax.set_title("Bahdanau Alignment Weights alpha_ij (T = 20 Sequence Reversal)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Alignment Weight")
plt.tight_layout()
plt.show()